# 🎯 Notebook 2: SLI, SLO, SLA — and the Error Budget

Three letters that confuse everyone:

- **SLI** — Service Level *Indicator*. The number you measure: e.g. *"% of requests that returned 200 in under 300ms"*.
- **SLO** — Service Level *Objective*. The target you set internally: e.g. *"99.9% over a rolling 28 days"*.
- **SLA** — Service Level *Agreement*. The contractual promise to a customer (with money involved if you miss it). Usually looser than the SLO.

The magic concept that comes out of this is the **error budget**: if your SLO is 99.9%, you are allowed to be bad 0.1% of the time. That 0.1% is a budget you can spend on risky deploys and experiments.

## Learning objectives
- Compute an SLI from raw request data.
- Track an error budget over time.
- See what "burning the budget too fast" looks like.

In [ ]:
import random, time
random.seed(42)

# Simulate one minute of traffic: 1000 requests with random outcomes.
requests = []
for _ in range(1000):
    latency = random.gauss(150, 80)
    success = (latency < 500) and (random.random() > 0.002)  # 99.8% success-ish
    requests.append({"latency_ms": max(0, latency), "ok": success})

good = sum(1 for r in requests if r["ok"] and r["latency_ms"] < 300)
total = len(requests)
sli = good / total
print(f"SLI (% requests OK and <300ms): {sli:.3%}")
print(f"SLO target: 99.0%  ->  {'✅ on track' if sli >= 0.99 else '❌ missed'}")

## 💰 The error budget

If your SLO is 99.0% over 30 days and you serve 1,000,000 requests in that window, your budget is 10,000 bad requests. After that, every additional bad request *costs* you the SLO.

Many teams use this rule: **if the budget is being burnt too fast, freeze risky deploys and focus on reliability**. If the budget is unused at the end of the window, you can take more risks (chaos tests, faster releases).

In [ ]:
TOTAL_REQUESTS = 1_000_000
SLO = 0.99
budget = TOTAL_REQUESTS * (1 - SLO)
print(f"30-day error budget: {int(budget):,} bad requests")

# Simulate 30 days, with one really bad day in the middle.
spent = 0
for day in range(1, 31):
    bad_today = 200 if day != 15 else 6000  # a big outage on day 15
    spent += bad_today
    pct = spent / budget
    bar = "█" * min(40, int(pct * 40))
    flag = "🚨" if pct > 1 else ("⚠️" if pct > 0.7 else "  ")
    print(f"day {day:2d}: spent {spent:6d}/{int(budget)}  {bar:40s} {flag}")

## 🤝 SLA vs SLO

People often ask, *"why are these different?"*

- **SLO** is what you aim for internally (e.g. 99.95%). You want some headroom.
- **SLA** is what you promise customers in a contract (e.g. 99.9%). Missing it costs you money — service credits, refunds, reputational damage.

Always set your **SLO tighter than your SLA**, so you start sweating *before* you owe customers money.